In [ ]:
import os
from dotenv import load_dotenv
from datasets import load_dataset

huggingface_key = "hugging_facekey"

if not huggingface_key:
    raise ValueError("HUGGINGFACE_KEY not found — add it to your .env file")

REPO_ID = "sujalgawas/jev-1min-stock-bars"

dataset = load_dataset(REPO_ID)

df = dataset["train"].to_pandas()

README.md:   0%|          | 0.00/747 [00:00<?, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/248M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/235M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8665852 [00:00<?, ? examples/s]

In [2]:
df.head()

,symbol,timestamp,open,high,low,close,volume,trade_count,vwap,log_return,bar_range,vwap_dist,log_vol_change
0,AAPL,2025-03-24 13:31:00+00:00,219.510,220.030,219.00,219.270,14112.0,140.0,219.611052,-0.002255,0.004697,-0.001553,-0.201411
1,AAPL,2025-03-24 13:32:00+00:00,219.070,219.565,219.06,219.380,8614.0,92.0,219.283090,0.000502,0.002302,0.000442,-0.493591
2,AAPL,2025-03-24 13:33:00+00:00,219.460,220.025,219.46,219.565,5156.0,78.0,219.774338,0.000843,0.002573,-0.000953,-0.513150
3,AAPL,2025-03-24 13:34:00+00:00,219.485,219.705,218.69,218.740,6135.0,91.0,219.077117,-0.003765,0.004640,-0.001539,0.173818
4,AAPL,2025-03-24 13:35:00+00:00,218.900,219.275,218.68,218.840,7409.0,84.0,218.943417,0.000457,0.002719,-0.000472,0.188657


## Multi-GPU training via the `accelerate` CLI

Quick note on the ask: you wrote "2x tpu ... double gpu" — I've built this for **2x GPU**
(e.g. Kaggle's "GPU T4 x2" option, or a paid Colab multi-GPU runtime), since that's what
"double gpu" points at and it's the far more common setup. An actual TPU runtime uses XLA
under the hood and needs different handling (dynamic indexing like `seq[:, ctx_idx, :] = proj`
doesn't behave the same way under TPU compilation) — ping me if you specifically meant TPU
and I'll adapt it.

Since notebook_launcher gave you trouble before, this goes through the CLI instead: everything
(model, dataset, training loop, HF Hub push) is one self-contained `.py` file, written to disk
from the cell below, then launched with `!accelerate launch`.

In [3]:
!pip install -q -U accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 7.0 MB/s eta 0:00:00a 0:00:01


In [ ]:
%%writefile train_jepa.py
"""
Standalone multi-GPU training script for ONE model: JEPADirectionModel.
Run via the accelerate CLI, not notebook_launcher:

    accelerate launch --multi_gpu --num_processes 2 --mixed_precision fp16 train_jepa.py \
        --hf-token <token>

There is exactly one model class and one checkpoint. The JEPA-style
self-supervised objective is how its encoder gets pretrained (phase 1), then
the SAME object's direction head is trained on top of that same encoder
(phase 2) -- it's one model throughout, saved once at the end as one file.
Inference loads this one class from this one file and calls
`model.predict_direction(x_past)` directly.
"""
from __future__ import annotations

import argparse
import copy
import json
import math
import os
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, random_split

from accelerate import Accelerator
from datasets import load_dataset
from huggingface_hub import HfApi, create_repo


# --------------------------------------------------------------------------
# Data
# --------------------------------------------------------------------------
class JEPATimeSeriesDataset(Dataset):
    """Slices past context window (X) and future target window (Y) on-the-fly."""

    def __init__(self, df, past_window=60, future_horizon=15,
                 feature_cols=("log_return", "bar_range", "vwap_dist", "log_vol_change")):
        self.past_window = past_window
        self.future_horizon = future_horizon
        self.feature_cols = list(feature_cols)
        self.valid_indices = []
        self.data_by_symbol = {}
        for symbol, group in df.groupby("symbol"):
            features = group[self.feature_cols].values.astype(np.float32)
            n_samples = len(features) - (past_window + future_horizon) + 1
            if n_samples > 0:
                self.data_by_symbol[symbol] = features
                for i in range(n_samples):
                    self.valid_indices.append((symbol, i))

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        symbol, start_i = self.valid_indices[idx]
        features = self.data_by_symbol[symbol]
        x_past = features[start_i: start_i + self.past_window]
        y_future = features[start_i + self.past_window: start_i + self.past_window + self.future_horizon]
        return torch.tensor(x_past), torch.tensor(y_future)


# --------------------------------------------------------------------------
# Building blocks
# --------------------------------------------------------------------------
class PatchEmbed(nn.Module):
    def __init__(self, n_features, patch_len, dim):
        super().__init__()
        self.patch_len = patch_len
        self.n_features = n_features
        self.proj = nn.Linear(patch_len * n_features, dim)

    def forward(self, x):
        B, L, Fdim = x.shape
        P = self.patch_len
        assert L % P == 0, f"window {L} not divisible by patch_len {P}"
        N = L // P
        x = x.reshape(B, N, P * Fdim)
        return self.proj(x)


class TransformerEncoder(nn.Module):
    def __init__(self, dim, num_layers, heads, mlp_ratio=4.0, dropout=0.0):
        super().__init__()
        layer = nn.TransformerEncoderLayer(
            d_model=dim, nhead=heads, dim_feedforward=int(dim * mlp_ratio),
            dropout=dropout, activation="gelu", batch_first=True, norm_first=True,
        )
        self.blocks = nn.TransformerEncoder(layer, num_layers=num_layers, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        return self.norm(self.blocks(x))


class LOBEncoder(nn.Module):
    def __init__(self, n_features, window, patch_len=8, dim=192, num_layers=6,
                 heads=6, mlp_ratio=4.0, dropout=0.0):
        super().__init__()
        self.n_patches = window // patch_len
        self.dim = dim
        self.patch_embed = PatchEmbed(n_features, patch_len, dim)
        self.pos_embed = nn.Parameter(torch.zeros(1, self.n_patches, dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.encoder = TransformerEncoder(dim, num_layers, heads, mlp_ratio, dropout)

    def embed_tokens(self, x):
        tokens = self.patch_embed(x)
        n = tokens.shape[1]
        # slice rather than assume n == self.n_patches -- lets this same encoder
        # run on just the past window (fewer patches) at inference time, same
        # positions 0..n-1 as during pretraining.
        return tokens + self.pos_embed[:, :n, :]

    def forward(self, x, idx=None):
        tokens = self.embed_tokens(x)
        if idx is not None:
            tokens = tokens[:, idx, :]
        return self.encoder(tokens)


class JEPAPredictor(nn.Module):
    """Only used during self-supervised pretraining (phase 1) -- not involved
    in direction prediction at inference time."""

    def __init__(self, dim, n_patches, pred_dim=128, num_layer=4, heads=4, mlp_ratio=4.0):
        super().__init__()
        self.input_proj = nn.Linear(dim, pred_dim)
        self.output_proj = nn.Linear(pred_dim, dim)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, pred_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, n_patches, pred_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.mask_token, std=0.02)
        self.encoder = TransformerEncoder(pred_dim, num_layer, heads, mlp_ratio)

    def forward(self, ctx_reps, ctx_idx, tgt_idx):
        proj = self.input_proj(ctx_reps)
        B = proj.shape[0]
        N = self.pos_embed.shape[1]
        seq = self.mask_token.to(proj.dtype).expand(B, N, -1).clone()
        seq[:, ctx_idx, :] = proj
        seq = seq + self.pos_embed.to(seq.dtype)
        seq = self.encoder(seq)
        return self.output_proj(seq[:, tgt_idx, :])


class DirectionHead(nn.Module):
    """Predict down/flat/up from pooled encoder representations."""

    def __init__(self, embedding_dim: int = 192):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(embedding_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 3),
        )

    def forward(self, representation: torch.Tensor) -> torch.Tensor:
        # representation: [batch, n_patches, embedding_dim]
        pooled = representation.mean(dim=1)
        return self.classifier(pooled)


def direction_labels(y_future: torch.Tensor, threshold: float = 0.0005) -> torch.Tensor:
    """Class indices: 0=down, 1=flat, 2=up, from cumulative future log-return."""
    cumulative_returns = y_future[:, :, 0].sum(dim=1)
    direction = torch.zeros_like(cumulative_returns)
    direction[cumulative_returns > threshold] = 1.0
    direction[cumulative_returns < -threshold] = -1.0
    return (direction + 1).long()


def momentum_schedule(step, total_steps, base=0.996, final=1.0):
    if total_steps <= 0:
        return base
    frac = min(1.0, step / total_steps)
    return final - (final - base) * 0.5 * (1 + np.cos(np.pi * frac))


# --------------------------------------------------------------------------
# THE model -- one class, everything lives inside it
# --------------------------------------------------------------------------
class JEPADirectionModel(nn.Module):
    """
    One model. `context_encoder` gets pretrained with a JEPA self-supervised
    objective (`jepa_loss`) using `target_encoder`/`predictor` as scaffolding
    for that objective only. `direction_head` sits on top of the SAME
    `context_encoder` and is what actually gets used at inference:

        model.predict_direction(x_past) -> (B, 3) logits

    `target_encoder` and `predictor` are pretraining machinery, not something
    a caller ever touches directly -- the only method that matters for
    prediction is `predict_direction`.
    """

    def __init__(self, n_features, window, patch_len=8, dim=192, num_layer=6,
                 heads=6, pred_dim=128, mlp_ratio=4.0, dropout=0.0,
                 class_mapping=None, direction_threshold=0.0005):
        super().__init__()
        self.config = dict(n_features=n_features, window=window, patch_len=patch_len,
                            dim=dim, num_layer=num_layer, heads=heads, pred_dim=pred_dim,
                            mlp_ratio=mlp_ratio, dropout=dropout)
        self.class_mapping = class_mapping or {"0": "down", "1": "flat", "2": "up"}
        self.direction_threshold = direction_threshold

        self.context_encoder = LOBEncoder(n_features, window, patch_len, dim, num_layer,
                                           heads, mlp_ratio, dropout)
        self.target_encoder = copy.deepcopy(self.context_encoder)
        for p in self.target_encoder.parameters():
            p.requires_grad_(False)
        self.n_patches = self.context_encoder.n_patches
        self.predictor = JEPAPredictor(dim, self.n_patches, pred_dim, num_layer, heads, mlp_ratio)
        self.direction_head = DirectionHead(embedding_dim=dim)
        self.dim = dim

    # ---- phase 1: self-supervised pretraining machinery ----
    @torch.no_grad()
    def update_target(self, momentum):
        for pe, pt in zip(self.context_encoder.parameters(), self.target_encoder.parameters()):
            pt.mul_(momentum).add_(pe.detach(), alpha=1.0 - momentum)

    def jepa_loss(self, x, ctx_idx, tgt_idx):
        with torch.no_grad():
            t_all = self.target_encoder(x, idx=None)
            tgt = t_all[:, tgt_idx, :]
            tgt = F.layer_norm(tgt, (tgt.shape[-1],))
        ctx = self.context_encoder(x, idx=ctx_idx)
        pred = self.predictor(ctx, ctx_idx, tgt_idx)
        loss = F.smooth_l1_loss(pred, tgt)
        with torch.no_grad():
            flat_t = tgt.reshape(-1, tgt.shape[-1])
            flat_p = pred.reshape(-1, pred.shape[-1])
            metrics = {"loss": loss.detach(), "tgt_std": flat_t.std(dim=0).mean(),
                       "pred_std": flat_p.std(dim=0).mean()}
        return loss, metrics

    # ---- the actual product ----
    def predict_direction(self, x_past: torch.Tensor) -> torch.Tensor:
        """Raw past-window features (B, past_window, n_features) -> (B, 3) logits."""
        reps = self.context_encoder(x_past, idx=None)
        return self.direction_head(reps)

    def direction_loss(self, x_past: torch.Tensor, y_future: torch.Tensor):
        logits = self.predict_direction(x_past)
        labels = direction_labels(y_future, self.direction_threshold)
        loss = F.cross_entropy(logits, labels)
        return loss, logits, labels

    def forward(self, x, ctx_idx, tgt_idx):
        """Phase-1 training entry point -- delegates to jepa_loss. Defining this
        (rather than only having jepa_loss as a plain method) matters: calling
        `model(x, ctx_idx, tgt_idx)` on the DDP-wrapped object routes through
        DistributedDataParallel.forward(), which is what sets up correct
        gradient-sync hooks. Calling `jepa_loss` directly on the unwrapped
        module would silently skip that -- each GPU would train an independent,
        divergent copy instead of syncing gradients."""
        return self.jepa_loss(x, ctx_idx, tgt_idx)

    @torch.inference_mode()
    def infer(self, x_past: torch.Tensor) -> dict:
        """Convenience single-sample inference. x_past: (past_window, n_features)
        or (1, past_window, n_features)."""
        if x_past.dim() == 2:
            x_past = x_past.unsqueeze(0)
        logits = self.predict_direction(x_past)
        probs = F.softmax(logits, dim=-1)[0]
        idx = int(probs.argmax().item())
        return {
            "direction": idx - 1,
            "direction_label": self.class_mapping[str(idx)],
            "confidence": float(probs[idx].item()),
            "probabilities": {self.class_mapping[str(i)]: float(probs[i]) for i in range(3)},
        }


# --------------------------------------------------------------------------
# Train helpers
# --------------------------------------------------------------------------
def param_groups(model, weight_decay):
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if p.ndim < 2 or "pos_embed" in name or "mask_token" in name:
            no_decay.append(p)
        else:
            decay.append(p)
    return [{"params": decay, "weight_decay": weight_decay},
            {"params": no_decay, "weight_decay": 0.0}]


def lr_factor(step, warmup_steps, total_steps):
    if step < warmup_steps:
        return (step + 1) / max(1, warmup_steps)
    frac = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1.0 + math.cos(math.pi * min(1.0, frac)))


def save_and_push(model, accelerator, config, out_dir, hf_repo_id, hf_token, api,
                   val_loss=None, direction_val_acc=None, filename="model.pt"):
    """The ONE checkpoint. Always the full model -- backbone + direction head
    together -- never split across files."""
    if not accelerator.is_main_process:
        return
    unwrapped = accelerator.unwrap_model(model)
    ckpt = dict(
        model=unwrapped.state_dict(),
        config=config,
        class_mapping=unwrapped.class_mapping,
        direction_threshold=unwrapped.direction_threshold,
        val_loss=val_loss,
        direction_val_accuracy=direction_val_acc,
    )
    path = os.path.join(out_dir, filename)
    torch.save(ckpt, path)
    api.upload_file(path_or_fileobj=path, path_in_repo=filename, repo_id=hf_repo_id, token=hf_token)


# --------------------------------------------------------------------------
# Main
# --------------------------------------------------------------------------
def main():
    p = argparse.ArgumentParser()
    p.add_argument("--dataset-repo", default="sujalgawas/jev-1min-stock-bars")
    p.add_argument("--past-window", type=int, default=60)
    p.add_argument("--future-horizon", type=int, default=15)
    p.add_argument("--patch-len", type=int, default=5)
    p.add_argument("--dim", type=int, default=192)
    p.add_argument("--pred-dim", type=int, default=192)
    p.add_argument("--num-layer", type=int, default=6)
    p.add_argument("--heads", type=int, default=6)
    p.add_argument("--epochs", type=int, default=30, help="JEPA pretraining epochs")
    p.add_argument("--direction-epochs", type=int, default=10)
    p.add_argument("--batch-size", type=int, default=32)
    p.add_argument("--lr", type=float, default=3e-4)
    p.add_argument("--direction-lr", type=float, default=1e-3)
    p.add_argument("--weight-decay", type=float, default=0.05)
    p.add_argument("--direction-weight-decay", type=float, default=1e-4)
    p.add_argument("--direction-threshold", type=float, default=0.0005)
    p.add_argument("--grad-clip", type=float, default=1.0)
    p.add_argument("--warmup-frac", type=float, default=0.05)
    p.add_argument("--ema-base", type=float, default=0.996)
    p.add_argument("--ema-final", type=float, default=1.0)
    p.add_argument("--out-dir", default="checkpoints")
    p.add_argument("--hf-repo-id", default="sujalgawas/jepa-trading-model")
    p.add_argument("--push-every", type=int, default=1, help="push the checkpoint every N epochs")
    p.add_argument("--num-workers", type=int, default=2)
    p.add_argument("--hf-token", default=None,
                    help="HF write token. Falls back to HUGGINGFACE_KEY env var if not passed.")
    args = p.parse_args()

    accelerator = Accelerator()
    device = accelerator.device

    feature_cols = ["log_return", "bar_range", "vwap_dist", "log_vol_change"]
    n_features = len(feature_cols)
    window = args.past_window + args.future_horizon
    assert args.past_window % args.patch_len == 0 and args.future_horizon % args.patch_len == 0, \
        "patch_len must evenly divide both past_window and future_horizon"

    os.makedirs(args.out_dir, exist_ok=True)

    hf_token = "your_hugging_face_token"
    api = None
    if accelerator.is_main_process:
        if not hf_token:
            raise RuntimeError(
                "No HF token found. Pass --hf-token directly on the launch command, "
                "or set the HUGGINGFACE_KEY env var before launching."
            )
        create_repo(repo_id=args.hf_repo_id, token=hf_token, exist_ok=True, private=True)
        api = HfApi(token=hf_token)

    if accelerator.is_main_process:
        accelerator.print("loading dataset...")
    raw = load_dataset(args.dataset_repo)
    df = raw["train"].to_pandas()

    dataset = JEPATimeSeriesDataset(df, args.past_window, args.future_horizon, feature_cols)
    length = len(dataset)
    train_size = int(length * 0.8)
    val_size = length - train_size
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(train_dataset, batch_size=args.batch_size, shuffle=True,
                               num_workers=args.num_workers, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=args.batch_size, shuffle=False,
                             num_workers=args.num_workers, pin_memory=True)

    config = dict(n_features=n_features, window=window, patch_len=args.patch_len,
                   dim=args.dim, num_layer=args.num_layer, heads=args.heads,
                   pred_dim=args.pred_dim, mlp_ratio=4.0, dropout=0.0)
    model = JEPADirectionModel(**config, direction_threshold=args.direction_threshold)

    optimizer = torch.optim.AdamW(param_groups(model, args.weight_decay), lr=args.lr, betas=(0.9, 0.95))

    n_ctx_patches = args.past_window // args.patch_len
    n_patches = model.n_patches

    model, optimizer, train_loader, val_loader = accelerator.prepare(
        model, optimizer, train_loader, val_loader
    )

    ctx_idx = torch.arange(n_ctx_patches, device=device)
    tgt_idx = torch.arange(n_ctx_patches, n_patches, device=device)

    steps_per_epoch = len(train_loader)
    total_steps = args.epochs * steps_per_epoch
    warmup_steps = max(1, int(args.warmup_frac * total_steps))

    # ======================================================================
    # Phase 1: self-supervised JEPA pretraining of context_encoder
    # ======================================================================
    @torch.no_grad()
    def validate_jepa():
        model.eval()
        losses = []
        for x_past, y_future in val_loader:
            x = torch.cat([x_past, y_future], dim=1)
            loss, _ = model(x, ctx_idx, tgt_idx)  # through DDP wrapper, same as training
            losses.append(accelerator.gather(loss.detach()).mean().item())
        model.train()
        return float(np.mean(losses)) if losses else float("nan")

    best_val = float("inf")
    step = 0
    model.train()
    t0 = time.time()
    lr = args.lr
    metrics = {"loss": torch.tensor(float("nan"))}

    for epoch in range(args.epochs):
        for x_past, y_future in train_loader:
            x = torch.cat([x_past, y_future], dim=1)

            lr = args.lr * lr_factor(step, warmup_steps, total_steps)
            for g in optimizer.param_groups:
                g["lr"] = lr

            loss, metrics = model(x, ctx_idx, tgt_idx)  # through DDP wrapper -- needed for grad sync
            optimizer.zero_grad(set_to_none=True)
            accelerator.backward(loss)
            accelerator.clip_grad_norm_(model.parameters(), args.grad_clip)
            optimizer.step()
            accelerator.unwrap_model(model).update_target(
                momentum_schedule(step, total_steps, args.ema_base, args.ema_final)
            )
            step += 1

        if accelerator.is_main_process:
            wps = step * args.batch_size / (time.time() - t0)
            accelerator.print(f"[pretrain] epoch {epoch+1}/{args.epochs}  loss {metrics['loss'].item():.4f}  "
                               f"lr {lr:.2e}  {wps:.0f} win/s")

        vloss = validate_jepa()
        accelerator.wait_for_everyone()

        if accelerator.is_main_process:
            accelerator.print(f"  [pretrain val] epoch {epoch+1}  loss {vloss:.4f}  (best {best_val:.4f})")
            improved = vloss < best_val
            if improved:
                best_val = vloss
            if (epoch + 1) % args.push_every == 0 or epoch == args.epochs - 1:
                save_and_push(model, accelerator, config, args.out_dir, args.hf_repo_id, hf_token, api,
                              val_loss=vloss, filename="model.pt")
                accelerator.print(f"  pushed model.pt -> https://huggingface.co/{args.hf_repo_id}")

    if accelerator.is_main_process:
        accelerator.print(f"pretraining done. best JEPA val loss: {best_val:.4f}")

    # ======================================================================
    # Phase 2: train direction_head on top of the SAME model's frozen encoder
    # ======================================================================
    if accelerator.is_main_process:
        accelerator.print("training direction head (same model, same checkpoint)...")

    backbone = accelerator.unwrap_model(model)
    for name, p_ in backbone.named_parameters():
        if not name.startswith("direction_head"):
            p_.requires_grad_(False)
    backbone.context_encoder.eval()

    head_optimizer = torch.optim.AdamW(
        backbone.direction_head.parameters(), lr=args.direction_lr,
        weight_decay=args.direction_weight_decay,
    )
    # wrap direction_head for correct multi-GPU gradient sync -- accelerator.prepare
    # wraps the SAME nn.Module object (parameter identity preserved), so
    # backbone.direction_head is updated in place through wrapped_head; we never
    # reassign backbone.direction_head itself, so it stays a plain, cleanly
    # state_dict-able module (no DDP "module." key prefixes) when saved later.
    wrapped_head, head_optimizer = accelerator.prepare(backbone.direction_head, head_optimizer)

    best_dir_acc = 0.0
    for epoch in range(args.direction_epochs):
        backbone.direction_head.train()
        total_loss, total_correct, total_examples = 0.0, 0, 0

        for x_past, y_future in train_loader:
            with torch.no_grad():
                reps = backbone.context_encoder(x_past, idx=None)  # frozen, no grad needed
            logits = wrapped_head(reps)  # through DDP wrapper -- needed for grad sync
            labels = direction_labels(y_future, args.direction_threshold)
            loss = F.cross_entropy(logits, labels)

            head_optimizer.zero_grad(set_to_none=True)
            accelerator.backward(loss)
            head_optimizer.step()

            total_loss += loss.item() * labels.shape[0]
            total_correct += (logits.argmax(dim=1) == labels).sum().item()
            total_examples += labels.shape[0]

        stats = torch.tensor([total_loss, total_correct, total_examples],
                              device=device, dtype=torch.float64)
        stats = accelerator.gather(stats.unsqueeze(0)).sum(dim=0)
        if accelerator.is_main_process:
            accelerator.print(f"[direction] epoch {epoch+1}/{args.direction_epochs} "
                               f"loss={(stats[0]/stats[2]).item():.4f} accuracy={(stats[1]/stats[2]).item():.4f}")

        backbone.direction_head.eval()
        val_loss, val_correct, val_examples = 0.0, 0, 0
        with torch.no_grad():
            for x_past, y_future in val_loader:
                loss, logits, labels = backbone.direction_loss(x_past, y_future)  # plain module, fine under no_grad
                val_loss += loss.item() * labels.shape[0]
                val_correct += (logits.argmax(dim=1) == labels).sum().item()
                val_examples += labels.shape[0]
        vstats = torch.tensor([val_loss, val_correct, val_examples],
                               device=device, dtype=torch.float64)
        vstats = accelerator.gather(vstats.unsqueeze(0)).sum(dim=0)

        if accelerator.is_main_process:
            val_acc = (vstats[1] / vstats[2]).item()
            accelerator.print(f"  [direction val] epoch {epoch+1} "
                               f"loss={(vstats[0]/vstats[2]).item():.4f} accuracy={val_acc:.4f}")
            if val_acc > best_dir_acc:
                best_dir_acc = val_acc
            if (epoch + 1) % args.push_every == 0 or epoch == args.direction_epochs - 1:
                # SAME filename as phase 1 -- there is only ever one checkpoint for this model
                save_and_push(model, accelerator, config, args.out_dir, args.hf_repo_id, hf_token, api,
                              val_loss=best_val, direction_val_acc=val_acc, filename="model.pt")
                accelerator.print(f"  pushed model.pt -> https://huggingface.co/{args.hf_repo_id}")

    if accelerator.is_main_process:
        accelerator.print(f"done. best direction val accuracy: {best_dir_acc:.4f}")


if __name__ == "__main__":
    main()

Writing train_jepa.py


In [5]:
!accelerate launch --multi_gpu --num_processes 2 --mixed_precision fp16 train_jepa.py \
    --epochs 30 \
    --batch-size 256 \
    --push-every 1 \
    --hf-repo-id sujalgawas/jepa-trading-model-direction

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
loading dataset...
^C
W0927 12:44:57.544000 145 torch/distributed/elastic/agent/server/api.py:739] Received 2 death signal, shutting down workers
W0927 12:44:57.546000 145 torch/distributed/elastic/multiprocessing/api.py:1010] Sending process 164 closing signal SIGINT
W0927 12:44:57.550000 145 torch/distributed/elastic/multiprocessing/api.py:1010] Sending process 165 closing signal SIGINT
